# MELD Majority-Class Baseline

CIS 5300 · Milestone 2

For each utterance, predict the most frequent of the seven emotion labels in the official MELD training split. Ties use the lowest label ID. The prediction script reads only the IDs from dev and test, then writes one JSONL object with `id` and `label` per utterance. The project scorer uses the gold labels to evaluate the predictions.

Run from the repository root in Jupyter, or open the notebook in Colab and select **Runtime → Run all**. A CPU runtime suffices.

In [1]:
import json
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

repo = Path.cwd()
if not (repo / "simple-baseline.py").is_file():
    repo = repo / "CIS5300_Final_Project"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/3XIA00/CIS5300_Final_Project.git", str(repo)],
            check=True,
        )
    if not (repo / "simple-baseline.py").is_file():
        # The baseline is available on this branch while the PR is under review.
        subprocess.run(
            ["git", "fetch", "--depth", "1", "origin", "codex/m2-simple-baseline"],
            cwd=repo, check=True,
        )
        subprocess.run(["git", "switch", "--detach", "FETCH_HEAD"], cwd=repo, check=True)

print("MELD baseline repository ready.")

MELD baseline repository ready.


## Generate predictions

The script counts the training labels, then writes the dev and test predictions under `output/simple-baseline/`.

In [2]:
for split in ("dev", "test"):
    log = subprocess.check_output(
        [sys.executable, "simple-baseline.py", "--split", split], cwd=repo, text=True,
    )
    print(log, end="")

print("\nFirst two test predictions:")
with (repo / "output/simple-baseline/test.jsonl").open(encoding="utf-8") as predictions:
    for _ in range(2):
        print(next(predictions).strip())

Training majority: neutral (4,710/9,989)
Wrote 1,109 predictions to output/simple-baseline/dev.jsonl
Training majority: neutral (4,710/9,989)
Wrote 2,610 predictions to output/simple-baseline/test.jsonl

First two test predictions:
{"id": "test:0:0", "label": "neutral"}
{"id": "test:0:1", "label": "neutral"}


## Evaluate the predictions

Weighted F1 weights each class by its gold support. Macro F1 gives each of the seven classes equal weight. The last row rescores the same test predictions after excluding 11 utterances flagged as suspected same-source overlap. All metrics use a 0–100 scale.

In [3]:
evaluations = [
    ("Dev", "dev", []),
    ("Full test", "test", []),
    ("Test excluding 11 overlaps", "test", ["--exclude-ids", "eval/suspected_overlap_test_ids.json"]),
]

header = f"{'Evaluation':<28} {'N':>5} {'Weighted F1':>12} {'Macro F1':>9} {'Accuracy':>9}"
print(header)
print("-" * len(header))
with TemporaryDirectory() as temp:
    metrics_path = Path(temp) / "metrics.json"
    for name, split, extra_args in evaluations:
        subprocess.run(
            [sys.executable, "score.py", "--gold", f"data/processed/{split}.jsonl",
             "--pred", f"output/simple-baseline/{split}.jsonl", "--json", str(metrics_path),
             *extra_args],
            cwd=repo, stdout=subprocess.DEVNULL, check=True,
        )
        scores = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(f"{name:<28} {scores['n']:>5,} {100 * scores['weighted_f1']:>12.2f} "
              f"{100 * scores['macro_f1']:>9.2f} {100 * scores['accuracy']:>9.2f}")

Evaluation                       N  Weighted F1  Macro F1  Accuracy
-------------------------------------------------------------------
Dev                          1,109        25.23      8.50     42.38
Full test                    2,610        31.27      9.28     48.12


Test excluding 11 overlaps   2,599        31.45      9.30     48.29


The training majority is `neutral`. Full-test accuracy equals the fraction of gold utterances labeled `neutral` (1,256 / 2,610), and the six other emotions each have F1 of zero.